<style>
/* Force hide code inputs ONLY in RISE slideshow */
.rise-enabled .input, 
.rise-enabled .input_area, 
.rise-enabled .jp-InputArea, 
.rise-enabled .jp-Cell-inputWrapper,
.rise-enabled .jp-Editor {
    display: none !important;
}
</style>

# Caminos Mínimos y la Matemática de la Relajación de Aristas
### Matemáticas Discretas: Algoritmos de Dijkstra y Bellman-Ford

**Objetivo:** Encontrar el camino de peso mínimo desde un origen $s$ a todos los demás vértices en un grafo dirigido y pesado $G=(V, E)$.

## I. Introducción al Problema del Camino Mínimo

### El significado del "Peso"
En matemática discreta, el peso $w(u,v)$ representa una métrica sobre la arista $(u,v)$, como distancia, tiempo o costo.

### El Desafío de los Ciclos
1. **Ciclos Positivos:** Un camino mínimo nunca contendrá un ciclo de peso positivo (podríamos eliminarlo para reducir el peso).
2. **Ciclos de Peso 0:** Pueden eliminarse sin cambiar el peso total.

**Teorema Fundamental:** En un grafo con $|V|$ vértices, cualquier camino mínimo simple contiene, como máximo, $|V|-1$ aristas.

## II. Concepto Central: "Relajación" Matemática

### Estimaciones de Camino Mínimo
Introducimos la variable $d[v]$, que representa nuestra **cota superior** actual del peso del camino mínimo desde el origen $s$ hasta el vértice $v$.

### La Desigualdad del Triángulo
Para cualquier arista $(u,v)$, el camino mínimo a $v$ no puede ser mayor que el camino mínimo a $u$ más el peso de la arista que los une:
$$\delta(s,v) \le \delta(s,u) + w(u,v)$$

### El Paso de Relajación
Probamos si podemos mejorar $d[v]$ pasando por $u$:
**Si** $d[v] > d[u] + w(u,v)$, **entonces** $d[v] = d[u] + w(u,v)$.

## III. Algoritmo de Dijkstra: La Estrategia Voraz (Greedy)

### Premisa Matemática
1. Solo funciona en grafos con pesos no negativos ($w(u,v) \ge 0$).
2. Mantiene un conjunto $S$ de vértices "finalizados".

### La Elección Voraz
En cada paso, seleccionamos el vértice $u \notin S$ con el valor de $d[u]$ más pequeño. 

**¿Por qué funciona?**
Al elegir el mínimo absoluto, garantizamos que no existe un camino más corto a través de otros vértices no visitados, ya que cualquier desvío añadiría pesos positivos, aumentando el costo total.

### Dijkstra: Pseudocódigo e Implementación

**Pseudocódigo (CLRS):**
```text
DIJKSTRA(G, w, s)
1  INITIALIZE-SINGLE-SOURCE(G, s)
2  S ← Ø
3  Q ← V[G]  // Cola de prioridad
4  while Q ≠ Ø
5      do u ← EXTRACT-MIN(Q)
6         S ← S U {u}
7         for each vertex v ∈ Adj[u]
8             do RELAX(u, v, w)
```

**Implementación en Python:**
```python
def dijkstra(V, Adj, s):
    d, pi = initialize_single_source(V, s)
    Q = [(d[v], v) for v in V]
    heapq.heapify(Q)
    while Q:
        du, u = heapq.heappop(Q)
        if du > d[u]: continue
        for v, weight in Adj.get(u, []):
            if relax(u, v, weight, d, pi):
                heapq.heappush(Q, (d[v], v))
    return d, pi
```

In [1]:
from exampleGraphs import get_clrs_graph, get_all_graphs
from interactiveAlgorithms import launch_case, launch_visualizer
import ipywidgets as widgets

G, s = get_clrs_graph()
launch_case(G, s, algo_name="Dijkstra")

### ¿Dónde falla Dijkstra?

Si existe una arista con **peso negativo**, el fundamento de la elección voraz colapsa. La suposición de que $d[u]$ es el mínimo absoluto se vuelve falsa, pues un camino a través de un nodo con estimación mayor podría reducirse drásticamente al cruzar una arista negativa.

## IV. Algoritmo de Bellman-Ford: Pesos Negativos

### Estrategia: Relajación Exhaustiva
En lugar de elegir, Bellman-Ford relaja **todas** las aristas del grafo. Repite este proceso exactamente $|V|-1$ veces.

### Intuición Matemática
**Propiedad de Relajación de Caminos:** Si relajamos las aristas de un camino mínimo en orden, encontraremos el camino mínimo.

Como un camino mínimo tiene a lo sumo $|V|-1$ aristas, realizar $|V|-1$ pasadas exhaustivas garantiza que cualquier secuencia de aristas que forme un camino mínimo será relajada en el orden correcto.

### Bellman-Ford: Pseudocódigo e Implementación

**Pseudocódigo (CLRS):**
```text
BELLMAN-FORD(G, w, s)
1  INITIALIZE-SINGLE-SOURCE(G, s)
2  for i ← 1 to |V[G]| - 1
3       do for each edge (u, v) ∈ E[G]
4              do RELAX(u, v, w)
5  for each edge (u, v) ∈ E[G]
6       do if d[v] > d[u] + w(u, v)
7             then return FALSE // Ciclo negativo detectado
8  return TRUE
```

**Implementación en Python:**
```python
def bellman_ford(V, E_weighted, s):
    d, pi = initialize_single_source(V, s)
    for i in range(len(V) - 1):
        for u, v, w in E_weighted:
            relax(u, v, w, d, pi)
    for u, v, w in E_weighted:
        if d[v] > d[u] + w:
            return False, d, pi # Ciclo negativo
    return True, d, pi
```

In [2]:
launch_case(G, s, algo_name="Bellman-Ford")

### Detección de Ciclos de Peso Negativo

Si después de $|V|-1$ pasadas, una arista todavía puede ser relajada ($d[v] > d[u] + w(u,v)$), es la **prueba matemática** de que existe un ciclo de peso negativo alcanzable desde el origen.

En tales grafos, el camino mínimo no está bien definido, ya que el peso tiende a $-\infty$.

## V. Conclusión y Comparativa

| Algoritmo | Estrategia | Pesos Negativos | Eficiencia |
| :--- | :--- | :--- | :--- |
| **Dijkstra** | Voraz (Greedy) | No permitidos | Alta (Eficaz) |
| **Bellman-Ford** | Dinámica / Exhaustiva | Permitidos | Media (Robusto) |

### Laboratorio Interactivo

In [3]:
graphs = get_all_graphs()
launch_case(graphs=graphs)